# 03. Silver Conformed & Governed Layer (SCD Type 2)

Cleanses, standardizes, and conforms data from Bronze into domain-driven schemas in `silver_prod`:
- **`silver_prod.customers.customers`**: PII-segregated, full name cleansing, domain parsing, **SCD Type 2 historical tracking**.
- **`silver_prod.products.products`**: Pricing tiers, discount logic, **SCD Type 2 historical tracking**.
- **`silver_prod.orders.orders`**: Timestamp casting, temporal attributes, window ranking analytics.
- **`silver_prod.region.region`**: Master geographic dimensional reference.
- **Governance**: Unity Catalog tags (`classification = 'PII'`) applied to sensitive customer schemas.


In [0]:
%run ./01_Config


In [0]:
import uuid
from pyspark.sql.functions import (
    col, split, concat_ws, trim, lower, when, lit, current_timestamp,
    sha2, coalesce, to_timestamp, year, month, quarter, dayofweek,
    row_number, rank, dense_rank, round
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

RUN_ID = str(uuid.uuid4())[:8]
print(f"[SILVER] Starting Silver layer processing. Run ID: {RUN_ID}")


### 1. Geography Domain: silver_prod.region.region


In [0]:
# Read Bronze Region & Standardize
df_bronze_region = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.region")
if "_rescued_data" in df_bronze_region.columns:
    df_bronze_region = df_bronze_region.drop("_rescued_data")

df_silver_region = (
    df_bronze_region
    .withColumn("region_name", trim(col("region_name")))
    .withColumn("country_code", trim(col("country_code")))
    .withColumn("currency", trim(col("currency")))
    .withColumn("_updated_timestamp", current_timestamp())
    .withColumn("_etl_run_id", lit(RUN_ID))
)

target_region_table = f"{SILVER_CATALOG}.region.region"

if DeltaTable.isDeltaTable(spark, f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_landing"): # fallback check
    pass

if spark.catalog.tableExists(target_region_table):
    tgt_region = DeltaTable.forName(spark, target_region_table)
    (tgt_region.alias("tgt")
     .merge(df_silver_region.alias("src"), "tgt.region_id = src.region_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_silver_region.write.format("delta").saveAsTable(target_region_table)

region_count = spark.table(target_region_table).count()
print(f"✓ Region conformed in {target_region_table}: {region_count} records.")


### 2. Product Domain: silver_prod.products.products (SCD Type 2)


In [0]:
# Read Bronze Products & Calculate Business Metrics
df_bronze_prod = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.products")
if "_rescued_data" in df_bronze_prod.columns:
    df_bronze_prod = df_bronze_prod.drop("_rescued_data")

df_incoming_prod = (
    df_bronze_prod
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("pricing_tier",
        when(col("unit_price") < 50.0, "Budget")
        .when(col("unit_price") < 250.0, "Mid-range")
        .otherwise("Premium"))
    .withColumn("standard_discount", round(col("unit_price") * 0.10, 2))
    .transform(lambda df: generate_record_hash(df, ["product_name", "category", "unit_price", "cost_price", "pricing_tier"]))
)

target_prod_table = f"{SILVER_CATALOG}.products.products"

# Delta Lake SCD Type 2 MERGE Pattern for Products
if not spark.catalog.tableExists(target_prod_table):
    print(f"[SCD2] Creating initial {target_prod_table} with active records...")
    (df_incoming_prod
     .withColumn("valid_from", current_timestamp())
     .withColumn("valid_to", lit(None).cast("timestamp"))
     .withColumn("is_current", lit(True))
     .withColumn("is_deleted", lit(False))
     .withColumn("etl_run_id", lit(RUN_ID))
     .write.format("delta").saveAsTable(target_prod_table))
else:
    target_prod = DeltaTable.forName(spark, target_prod_table)
    df_curr_prod = target_prod.toDF().filter("is_current = true")
    
    # Find modified records (existing active record with changed hash)
    staged_updates = (
        df_incoming_prod.alias("inc")
        .join(df_curr_prod.alias("cur"), on="product_id", how="inner")
        .filter("inc.record_hash != cur.record_hash")
        .select("inc.*")
    )
    
    # 1. Close out expired records (set is_current = false, valid_to = current_timestamp)
    (target_prod.alias("tgt")
     .merge(staged_updates.alias("upd"), "tgt.product_id = upd.product_id AND tgt.is_current = true")
     .whenMatchedUpdate(set={
         "is_current": lit(False),
         "valid_to": current_timestamp(),
         "etl_run_id": lit(RUN_ID)
     })
     .execute())
     
    # 2. Insert new records & new versions of modified records
    df_to_insert = (
        df_incoming_prod.alias("inc")
        .join(df_curr_prod.alias("cur"), on="product_id", how="left")
        .filter("cur.product_id IS NULL OR inc.record_hash != cur.record_hash")
        .select("inc.*")
        .withColumn("valid_from", current_timestamp())
        .withColumn("valid_to", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .withColumn("is_deleted", lit(False))
        .withColumn("etl_run_id", lit(RUN_ID))
    )
    
    (target_prod.alias("tgt")
     .merge(df_to_insert.alias("ins"), "1 = 0")  # Force insert
     .whenNotMatchedInsertAll()
     .execute())

prod_count = spark.table(target_prod_table).count()
prod_active = spark.table(target_prod_table).filter("is_current = true").count()
print(f"✓ Products SCD Type 2 conformed in {target_prod_table}: {prod_count} total rows ({prod_active} active).")


### 3. Customer Domain: silver_prod.customers.customers (SCD Type 2 & PII Governance)


In [0]:
# Read Bronze Customers, Clean Name & Extract Email Domain
df_bronze_cust = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.customers")
if "_rescued_data" in df_bronze_cust.columns:
    df_bronze_cust = df_bronze_cust.drop("_rescued_data")

df_incoming_cust = (
    df_bronze_cust
    .withColumn("fullname", concat_ws(" ", trim(col("first_name")), trim(col("last_name"))))
    .drop("first_name", "last_name")
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("domain", split(col("email"), "@").getItem(1))
    .withColumn("phone_number", trim(col("phone_number")))
    .transform(lambda df: generate_record_hash(df, ["fullname", "email", "phone_number", "region_id"]))
)

target_cust_table = f"{SILVER_CATALOG}.customers.customers"

# Delta Lake SCD Type 2 MERGE Pattern for Customers
if not spark.catalog.tableExists(target_cust_table):
    print(f"[SCD2] Creating initial {target_cust_table} with active records...")
    (df_incoming_cust
     .withColumn("valid_from", current_timestamp())
     .withColumn("valid_to", lit(None).cast("timestamp"))
     .withColumn("is_current", lit(True))
     .withColumn("is_deleted", lit(False))
     .withColumn("etl_run_id", lit(RUN_ID))
     .write.format("delta").saveAsTable(target_cust_table))
else:
    target_cust = DeltaTable.forName(spark, target_cust_table)
    df_curr_cust = target_cust.toDF().filter("is_current = true")
    
    # Find modified customers
    staged_cust_updates = (
        df_incoming_cust.alias("inc")
        .join(df_curr_cust.alias("cur"), on="customer_id", how="inner")
        .filter("inc.record_hash != cur.record_hash")
        .select("inc.*")
    )
    
    # 1. Expire outdated customer versions
    (target_cust.alias("tgt")
     .merge(staged_cust_updates.alias("upd"), "tgt.customer_id = upd.customer_id AND tgt.is_current = true")
     .whenMatchedUpdate(set={
         "is_current": lit(False),
         "valid_to": current_timestamp(),
         "etl_run_id": lit(RUN_ID)
     })
     .execute())
     
    # 2. Insert new customers & new active versions of updated customers
    df_cust_to_insert = (
        df_incoming_cust.alias("inc")
        .join(df_curr_cust.alias("cur"), on="customer_id", how="left")
        .filter("cur.customer_id IS NULL OR inc.record_hash != cur.record_hash")
        .select("inc.*")
        .withColumn("valid_from", current_timestamp())
        .withColumn("valid_to", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
        .withColumn("is_deleted", lit(False))
        .withColumn("etl_run_id", lit(RUN_ID))
    )
    
    (target_cust.alias("tgt")
     .merge(df_cust_to_insert.alias("ins"), "1 = 0")
     .whenNotMatchedInsertAll()
     .execute())

# Apply Unity Catalog Governance Security Tags (PII)
try:
    spark.sql(f"ALTER TABLE {target_cust_table} SET TAGS ('classification' = 'PII', 'sensitivity' = 'Restricted')")
    print(f"✓ Applied Unity Catalog PII Governance tags to {target_cust_table}")
except Exception as e:
    print(f"Note on tags: {e}")

cust_count = spark.table(target_cust_table).count()
cust_active = spark.table(target_cust_table).filter("is_current = true").count()
print(f"✓ Customers SCD Type 2 conformed in {target_cust_table}: {cust_count} total ({cust_active} active).")


### 4. Sales Domain: silver_prod.orders.orders


In [0]:
# Read Bronze Orders, Parse Timestamps, Temporal Attributes & Ranking Analytics
df_bronze_orders = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.orders")
if "_rescued_data" in df_bronze_orders.columns:
    df_bronze_orders = df_bronze_orders.drop("_rescued_data")

df_clean_orders = (
    df_bronze_orders
    .withColumn("order_date", to_timestamp(col("order_date")))
    .withColumn("order_year", year(col("order_date")))
    .withColumn("order_month", month(col("order_date")))
    .withColumn("order_quarter", quarter(col("order_date")))
    .withColumn("order_day_of_week", dayofweek(col("order_date")))
)

# Window Ranking analytics: row_number, rank, dense_rank over year ordered by total_amount
year_window = Window.partitionBy("order_year").orderBy(col("total_amount").desc())

df_ranked_orders = (
    df_clean_orders
    .withColumn("row_number_rank", row_number().over(year_window))
    .withColumn("rank_num", rank().over(year_window))
    .withColumn("dense_rank_num", dense_rank().over(year_window))
    .withColumn("_updated_timestamp", current_timestamp())
    .withColumn("_etl_run_id", lit(RUN_ID))
)

target_orders_table = f"{SILVER_CATALOG}.orders.orders"

if spark.catalog.tableExists(target_orders_table):
    tgt_orders = DeltaTable.forName(spark, target_orders_table)
    (tgt_orders.alias("tgt")
     .merge(df_ranked_orders.alias("src"), "tgt.order_id = src.order_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_ranked_orders.write.format("delta").saveAsTable(target_orders_table)

orders_count = spark.table(target_orders_table).count()
print(f"✓ Orders conformed in {target_orders_table}: {orders_count} records.")


### 5. Silver Integrity Assertions & Execution Summary


In [0]:
# Data Quality Assertions
assert spark.table(f"{SILVER_CATALOG}.customers.customers").filter("customer_id IS NULL").count() == 0, "Null customer_id found!"
assert spark.table(f"{SILVER_CATALOG}.products.products").filter("product_id IS NULL").count() == 0, "Null product_id found!"
assert spark.table(f"{SILVER_CATALOG}.orders.orders").filter("order_id IS NULL").count() == 0, "Null order_id found!"
assert spark.table(f"{SILVER_CATALOG}.region.region").filter("region_id IS NULL").count() == 0, "Null region_id found!"

silver_summary = [
    ("silver_prod.region.region", spark.table(f"{SILVER_CATALOG}.region.region").count(), "SCD Type 1 Conformed"),
    ("silver_prod.products.products", spark.table(f"{SILVER_CATALOG}.products.products").count(), f"SCD Type 2 ({spark.table(f'{SILVER_CATALOG}.products.products').filter('is_current = true').count()} active)"),
    ("silver_prod.customers.customers", spark.table(f"{SILVER_CATALOG}.customers.customers").count(), f"SCD Type 2 ({spark.table(f'{SILVER_CATALOG}.customers.customers').filter('is_current = true').count()} active, PII Tagged)"),
    ("silver_prod.orders.orders", spark.table(f"{SILVER_CATALOG}.orders.orders").count(), "Cleaned Transactions + Window Ranks")
]
display(spark.createDataFrame(silver_summary, ["Table", "RowCount", "GovernanceNote"]))
